# Notion as a visual dispatch board for Claude agents

Most agent systems give you logs. This notebook shows you how to give yourself a **Kanban board**: a Notion Tasks database where every in-flight agent job is a card you can read, approve, annotate, and close — from your phone if you want.

The pattern is called **agency-os**. The core idea is simple: Notion is the source of truth for what agents should do, and Claude is the executor. You write tasks on the board, approve them on the board, and results come back to the board. The agent loop lives entirely in Claude Code (or any Python process); Notion is never in the critical path of execution — it is the audit trail and the human-control surface.

This notebook walks through the full loop end-to-end:

1. **Scaffold** a minimal Tasks database in your Notion workspace.
2. **Create a Suggestion** — an idea that is not yet approved.
3. **Discuss and approve** the task, moving it to `To-Do`.
4. **Dispatch** a Claude subagent to execute it.
5. **Close the loop**: the subagent writes its Result Link back to the Notion card.

### Why a board instead of a queue

A pure queue (a list of prompts to run) answers "what did the agent do?" A board answers "what is the agent doing *right now*, and can I redirect it?" That second question matters for teams and for any workflow where human judgment is part of the product — content review, client deliverables, anything that should not auto-ship.

The `Status` column on the board is the gate:

```
Suggestion → Discussion → To-Do → In Progress → Done
```

The agent is only allowed to pick up rows in `To-Do`. Approving a task is the human's explicit authorization. Nothing runs that you haven't said yes to.

### What you need

- An Anthropic API key (`ANTHROPIC_API_KEY`).
- A Notion integration token (`NOTION_KEY`) with write access to a page you own. [Create one here](https://www.notion.so/my-integrations).
- Python 3.11+.

In [ ]:
%%capture
%pip install -q anthropic python-dotenv requests

In [ ]:
import os
import json
import time
import requests
from datetime import datetime, timezone
from dotenv import load_dotenv
import anthropic

load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
NOTION_KEY = os.environ["NOTION_KEY"]
# A Notion page ID where the Tasks database will be created.
# Right-click any Notion page → Copy link → the last 32-char hex string is the ID.
PARENT_PAGE_ID = os.environ.get("NOTION_PARENT_PAGE_ID", "")

NOTION_VERSION = "2022-06-28"
NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_KEY}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

print(f"Anthropic client ready. Model: {MODEL}")
print(f"Notion integration connected: {'yes' if NOTION_KEY else 'NO — set NOTION_KEY'}")

## 1. Scaffold a Tasks database

We create a minimal Notion database with the columns the dispatch loop depends on:

| Property | Type | Purpose |
|---|---|---|
| `Title` | title | What the task is |
| `Status` | status | Controls which tasks agents can pick up |
| `Exec` | select | `Agent` = runnable by the dispatch loop; `Human` = operator-only |
| `Result Link` | url | Where the agent writes its output link |
| `Priority` | select | 1 (urgent) to 4 (nice-to-have) |

In production you would add `Corpus`, `Effort`, `Dependencies`, and `Parent Task` — but these five are enough to demonstrate the full dispatch loop.

In [ ]:
def create_tasks_database(parent_page_id: str) -> dict:
    """Scaffold a minimal Tasks database under parent_page_id."""
    payload = {
        "parent": {"type": "page_id", "page_id": parent_page_id},
        "title": [{"type": "text", "text": {"content": "Tasks"}}],
        "properties": {
            "Title": {"title": {}},
            "Status": {
                "status": {
                    "options": [
                        {"name": "Suggestion", "color": "gray"},
                        {"name": "Discussion", "color": "yellow"},
                        {"name": "To-Do", "color": "blue"},
                        {"name": "In Progress", "color": "orange"},
                        {"name": "Done", "color": "green"},
                        {"name": "Killed", "color": "red"},
                    ],
                    "groups": [
                        {"name": "Not started", "color": "gray", "option_ids": []},
                        {"name": "In progress", "color": "blue", "option_ids": []},
                        {"name": "Complete", "color": "green", "option_ids": []},
                    ],
                }
            },
            "Exec": {
                "select": {
                    "options": [
                        {"name": "none", "color": "gray"},
                        {"name": "Agent", "color": "blue"},
                        {"name": "Human", "color": "orange"},
                    ]
                }
            },
            "Result Link": {"url": {}},
            "Priority": {
                "select": {
                    "options": [
                        {"name": "1", "color": "red"},
                        {"name": "2", "color": "orange"},
                        {"name": "3", "color": "yellow"},
                        {"name": "4", "color": "gray"},
                    ]
                }
            },
        },
    }
    resp = requests.post(
        "https://api.notion.com/v1/databases",
        headers=NOTION_HEADERS,
        json=payload,
    )
    resp.raise_for_status()
    return resp.json()


if not PARENT_PAGE_ID:
    raise ValueError(
        "Set NOTION_PARENT_PAGE_ID to a Notion page ID where the database will be created. "
        "Right-click any Notion page → Copy link → extract the last 32-char hex string."
    )

db = create_tasks_database(PARENT_PAGE_ID)
DATABASE_ID = db["id"]
print(f"Tasks database created: {db['url']}")
print(f"Database ID: {DATABASE_ID}")

## 2. Create a Suggestion

A Suggestion is an idea that is not yet approved for execution. In production you would add it via `suggest` (which deduplicates against existing tasks). Here we create one directly via the Notion API.

The task we're creating: **"Write a haiku about agency-os"** — a tiny, verifiable task that lets us confirm the full loop works end-to-end.

In [ ]:
def create_task(
    database_id: str,
    title: str,
    status: str = "Suggestion",
    exec_mode: str = "none",
    priority: str = "4",
    description: str = "",
) -> dict:
    """Create a task row in the Tasks database."""
    properties = {
        "Title": {"title": [{"text": {"content": title}}]},
        "Status": {"status": {"name": status}},
        "Exec": {"select": {"name": exec_mode}},
        "Priority": {"select": {"name": priority}},
    }
    children = []
    if description:
        children = [
            {
                "object": "block",
                "type": "paragraph",
                "paragraph": {
                    "rich_text": [{"type": "text", "text": {"content": description}}]
                },
            }
        ]
    payload = {
        "parent": {"database_id": database_id},
        "properties": properties,
    }
    if children:
        payload["children"] = children
    resp = requests.post(
        "https://api.notion.com/v1/pages",
        headers=NOTION_HEADERS,
        json=payload,
    )
    resp.raise_for_status()
    return resp.json()


suggestion = create_task(
    database_id=DATABASE_ID,
    title="Write a haiku about agency-os",
    status="Suggestion",
    exec_mode="none",
    priority="4",
    description=(
        "Write a 5-7-5 haiku that captures the core idea of agency-os: "
        "agents working, humans watching, tasks moving across the board."
    ),
)
TASK_ID = suggestion["id"]
print(f"Suggestion created: {suggestion['url']}")
print(f"Task ID: {TASK_ID}")
print(f"Status: {suggestion['properties']['Status']['status']['name']}")

## 3. Discuss, then approve

In practice, **discuss** is where you would log clarifying questions and answers on the task card — narrowing the scope, adding acceptance criteria, deciding on the approach. The board makes that conversation visible to everyone on the team.

Once the task is clear, **approve** moves it to `To-Do` and sets `Exec=Agent`. That two-step promotion is the authorization gate: the dispatch loop only picks up rows that are *both* `To-Do` *and* `Exec=Agent`. You can have a hundred To-Do tasks with `Exec=Human` that the agent will never touch.

Here we simulate both steps: add a discussion comment, then flip the row to `To-Do` + `Exec=Agent`.

In [ ]:
def update_task_properties(page_id: str, properties: dict) -> dict:
    """Update a Notion page's properties."""
    resp = requests.patch(
        f"https://api.notion.com/v1/pages/{page_id}",
        headers=NOTION_HEADERS,
        json={"properties": properties},
    )
    resp.raise_for_status()
    return resp.json()


def append_block(page_id: str, text: str, heading: str = "") -> None:
    """Append a paragraph (optionally preceded by a heading) to a page."""
    blocks = []
    if heading:
        blocks.append({
            "object": "block",
            "type": "heading_3",
            "heading_3": {
                "rich_text": [{"type": "text", "text": {"content": heading}}]
            },
        })
    blocks.append({
        "object": "block",
        "type": "paragraph",
        "paragraph": {
            "rich_text": [{"type": "text", "text": {"content": text}}]
        },
    })
    resp = requests.patch(
        f"https://api.notion.com/v1/blocks/{page_id}/children",
        headers=NOTION_HEADERS,
        json={"children": blocks},
    )
    resp.raise_for_status()


# Step 1: Move to Discussion and log a clarification note.
update_task_properties(TASK_ID, {"Status": {"status": {"name": "Discussion"}}})
now = datetime.now(timezone.utc).strftime("%Y-%m-%d")
append_block(
    TASK_ID,
    text=(
        f"{now} — clarification: haiku should be 5-7-5 syllables. "
        "Theme: tasks flowing from left to right across the board. "
        "Acceptance: 3 lines, correct syllable count."
    ),
    heading="Discussion log",
)
print("Status → Discussion. Discussion note appended.")

# Step 2: Approve — move to To-Do and mark Exec=Agent.
update_task_properties(
    TASK_ID,
    {
        "Status": {"status": {"name": "To-Do"}},
        "Exec": {"select": {"name": "Agent"}},
        "Priority": {"select": {"name": "2"}},
    },
)
print("Status → To-Do, Exec → Agent. Task is now dispatchable.")

## 4. Query the runnable queue

The dispatch loop scans for rows where `Status=To-Do` AND `Exec=Agent`. This is the only gate the agent enforces. Any row that doesn't meet both conditions is invisible to the agent — it will never be touched without explicit human promotion.

In production you would also check `Dependencies` (all must be `Done`) and run a topological sort to find parallel vs. sequential stages. Here we keep the filter minimal.

In [ ]:
def query_runnable_tasks(database_id: str) -> list[dict]:
    """Return all rows with Status=To-Do and Exec=Agent."""
    payload = {
        "filter": {
            "and": [
                {"property": "Status", "status": {"equals": "To-Do"}},
                {"property": "Exec", "select": {"equals": "Agent"}},
            ]
        },
        "sorts": [{"property": "Priority", "direction": "ascending"}],
    }
    resp = requests.post(
        f"https://api.notion.com/v1/databases/{database_id}/query",
        headers=NOTION_HEADERS,
        json=payload,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


runnable = query_runnable_tasks(DATABASE_ID)
print(f"Runnable tasks: {len(runnable)}")
for t in runnable:
    title = t["properties"]["Title"]["title"][0]["plain_text"]
    priority = t["properties"]["Priority"]["select"]["name"]
    print(f"  [{priority}] {title} — {t['url']}")

## 5. Dispatch the execution agent

For each runnable task the dispatcher:

1. Flips the row to `In Progress` — this is the live-status signal on the board.
2. Builds a brief from the task title and any description blocks on the page.
3. Calls Claude to execute the brief. The model choice is made at dispatch time, not pre-tagged on the row — you pick Haiku for mechanical tasks, Sonnet for judgment-bearing ones, Opus for strategic work.
4. Writes the result back to the row's `Result Link` and flips status to `Done`.

This is the **human-in-the-loop model at full resolution**: humans decide *what* (Suggestion → Discussion), *when* (approve → To-Do), and can watch *live* (In Progress card on the board). The agent decides *how*.

In [ ]:
def get_task_description(page_id: str) -> str:
    """Extract plain-text content from a Notion page's blocks."""
    resp = requests.get(
        f"https://api.notion.com/v1/blocks/{page_id}/children",
        headers=NOTION_HEADERS,
    )
    resp.raise_for_status()
    parts = []
    for block in resp.json().get("results", []):
        btype = block.get("type", "")
        rich = block.get(btype, {}).get("rich_text", [])
        text = "".join(r.get("plain_text", "") for r in rich)
        if text:
            parts.append(text)
    return "\n".join(parts)


def pick_model(title: str) -> str:
    """Heuristic: pick Haiku for mechanical tasks, Sonnet for judgment-bearing ones."""
    haiku_signals = ["submit", "pr ", "file ", "post ", "log ", "update ", "haiku"]
    if any(s in title.lower() for s in haiku_signals):
        return "claude-haiku-4-5"  # fast and cheap for simple tasks
    return MODEL  # Sonnet for everything substantive


def execute_task(task: dict) -> dict:
    """Run a single task through Claude and return {status, result_link, output}."""
    page_id = task["id"]
    title = task["properties"]["Title"]["title"][0]["plain_text"]
    description = get_task_description(page_id)
    model = pick_model(title)

    # Flip to In Progress — visible on the board immediately.
    update_task_properties(page_id, {"Status": {"status": {"name": "In Progress"}}})
    print(f"[{model}] Starting: {title}")

    system = (
        "You are an execution agent. Complete the task in the user message exactly as described. "
        "Return ONLY the final deliverable — no preamble, no explanation."
    )
    user_content = f"Task: {title}\n\nContext:\n{description}" if description else f"Task: {title}"

    try:
        msg = anthropic_client.messages.create(
            model=model,
            max_tokens=512,
            system=system,
            messages=[{"role": "user", "content": user_content}],
        )
        output = msg.content[0].text.strip()
        status = "done"
    except Exception as exc:
        output = f"failed: {exc}"
        status = "failed"

    return {"page_id": page_id, "title": title, "model": model, "status": status, "output": output}


def close_task(result: dict) -> None:
    """Write the result back to Notion and close the row."""
    page_id = result["page_id"]
    if result["status"] == "done":
        final_status = "Done"
        # Store the output as the result link (for text outputs: a data URI is impractical;
        # in production you would upload to a paste service or your own storage).
        # Here we store a short note in the done log and leave Result Link empty.
        now = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        append_block(
            page_id,
            text=f"{now}: completed by agent ({result['model']})\n\nOutput:\n{result['output']}",
            heading="Done log",
        )
    else:
        final_status = "Discussion"
        now = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        append_block(
            page_id,
            text=f"{now}: {result['output']}",
            heading="Discussion log",
        )

    update_task_properties(page_id, {"Status": {"status": {"name": final_status}}})
    print(f"  → status: {final_status}")


# Dispatch all runnable tasks.
results = []
for task in runnable:
    result = execute_task(task)
    close_task(result)
    results.append(result)
    print(f"  output: {result['output'][:200]}")
    print()

## 6. Verify: read back the closed task

The board should now show the card as `Done` with the agent's output in the Done log. Let's confirm by re-fetching the row.

In [ ]:
def get_task(page_id: str) -> dict:
    resp = requests.get(
        f"https://api.notion.com/v1/pages/{page_id}",
        headers=NOTION_HEADERS,
    )
    resp.raise_for_status()
    return resp.json()


for result in results:
    page = get_task(result["page_id"])
    title = page["properties"]["Title"]["title"][0]["plain_text"]
    status = page["properties"]["Status"]["status"]["name"]
    print(f"Task: {title}")
    print(f"Status: {status}")
    print(f"URL: {page['url']}")
    assert status == "Done", f"Expected Done, got {status}"
    print("Assertion passed: task is Done.")
    print()

## What you built

A five-cell loop that demonstrates the full agency-os pattern:

1. **Scaffold** — one database, five columns, zero dependencies outside the Notion API and the Anthropic SDK.
2. **Create** — tasks start as Suggestions, not runnable.
3. **Discuss + approve** — two explicit steps to promote a task. The gate is the `To-Do + Agent` combination; nothing runs without it.
4. **Dispatch** — the agent reads the brief from the card, executes it, and writes the result back.
5. **Verify** — the result lives on the board, not in a log file.

### What to build next

- **Parent/child tasks** — add a `Parent Task` self-relation to the database, and implement dependency gating (skip rows whose deps are not yet Done).
- **Corpus columns** — add a `Corpus` select to group tasks by domain ("Infrastructure", "Content", "Community") and filter the dispatch queue by corpus.
- **Recurring tasks** — set `Type=recurring` and `Cadence=weekly`; the loop re-queues the row instead of marking it Done.
- **Parallel stages** — build a DAG from the dependency graph, assign stage numbers, and fan out within a stage using `asyncio.gather`.
- **Claude Code skill** — wrap this loop as a Claude Code slash command so any team member can run `/dispatch` from the terminal and see the board update in real time.

The [agency-os repository](https://github.com/AutomateLab-tech/agency-os) ships all of these as production-ready skills, including a full sync layer that keeps a local JSON mirror of the Notion database for low-latency reads.